In [1]:
# This is a walkthrough of self-contained implmentation of reverse mode auto-differentiation
# first lets import the relevant packages and our global states
# NOTE: we constructed the Variable ( or this tutorials Neuron class in the file PyTorchAutoGradEx - it has notes and comments that explain
# why things are the way the are
from typing import List
import os
import os, sys
dir2 = os.path.abspath('')
dir1 = os.path.dirname(dir2)
if not dir1 in sys.path: sys.path.append(dir1)
from PyTorchAutoGradEx import Variable, TapeEntry, grad, reset_tape, gradient_tape
import torch


In [2]:
# We will now use our packages to calculate some gradients
a_global, b_global = torch.rand(4), torch.rand(4)
def simple(a,b):
    t = a + b
    return t*b


reset_tape() # to clear any computes
a = Variable.constant(a_global, name='a')
b = Variable.constant(b_global, name='b')
loss = simple(a,b)
da, db = grad(loss, [a,b])
print("da", da)
print("db", db)

a = tensor([0.2968, 0.7435, 0.0106, 0.0149])
b = tensor([0.7268, 0.8585, 0.7843, 0.6329])
v0 = a + b
v1 = v0 * b
dv1 ------------------------
v3 = v2 * b
v4 = v2 * v0
dv1_dv1 = v2
dv1_dv0 = v3
dv1_db = v4
------------------------
v5 = v4 + v3
dv1_dv1 = v2
dv1_dv0 = v3
dv1_db = v5
dv1_da = v3
------------------------
da tensor([0.7268, 0.8585, 0.7843, 0.6329])
db tensor([1.7504, 2.4605, 1.5791, 1.2808])


In [3]:
# NOTE that our propgate definitions takes in Variables not Tensors
# this is done because we could take the gradients ... of gradients
def run_gradients(my_fn, second_loss=True):
    reset_tape()
    a = Variable.constant(a_global, name='a')
    b = Variable.constant(b_global, name='b')

    # our first loss
    L0 = (my_fn(a, b)).sum(name='L0')

    # compute derivatives of our inputs
    dL0_da, dL0_db = grad(L0, [a, b])
    if not second_loss:
      return dL0_da, dL0_db

    # now lets compute the squared L2 norm of our derivatives
    # L2 Norm is the euclidien length ( sqrt( x**2 + y**2 ) )
    L1 = (dL0_da*dL0_da + dL0_db*dL0_db).sum(name='L1')

    # and take the gradient of that.
    # notice there are two losses involved.
    dL1_da, dL1_db = grad(L1, [a, b])
    return dL1_da, dL1_db

da, db = run_gradients(simple)
print("da", da)
print("db", db)
# Notice that we see an increase in the number of calls - because we still have in our example the first set of gradient computations
# done by the L0 loss - which makes sense since - we still have to consider all of the ways the gradient flows all the way from L1 to inputs
# a and b -> which makes sense - if you're doing the gradient of a gradient - you effectively have to run that calculation 4 times
# so really the whole training sequence for L2 is forward-L1, backward-L1, forward-L2, bacward-L2 AND THEN ANOTHER backward-L1
# although technically possible - we can see we blow up computational costs

a = tensor([0.2968, 0.7435, 0.0106, 0.0149])
b = tensor([0.7268, 0.8585, 0.7843, 0.6329])
v0 = a + b
v1 = v0 * b
L0 = v1.sum()
dL0 ------------------------
v3 = v2.expand(4)
dL0_dL0 = v2
dL0_dv1 = v3
------------------------
v4 = v3 * b
v5 = v3 * v0
dL0_dL0 = v2
dL0_dv1 = v3
dL0_dv0 = v4
dL0_db = v5
------------------------
v6 = v5 + v4
dL0_dL0 = v2
dL0_dv1 = v3
dL0_dv0 = v4
dL0_db = v6
dL0_da = v4
------------------------
v7 = v4 * v4
v8 = v6 * v6
v9 = v7 + v8
L1 = v9.sum()
dL1 ------------------------
v11 = v10.expand(4)
dL1_dL1 = v10
dL1_dv9 = v11
------------------------
dL1_dL1 = v10
dL1_dv9 = v11
dL1_dv7 = v11
dL1_dv8 = v11
------------------------
v12 = v11 * v6
v13 = v11 * v6
v14 = v12 + v13
dL1_dL1 = v10
dL1_dv9 = v11
dL1_dv7 = v11
dL1_dv8 = v11
dL1_dv6 = v14
------------------------
v15 = v11 * v4
v16 = v11 * v4
v17 = v15 + v16
dL1_dL1 = v10
dL1_dv9 = v11
dL1_dv7 = v11
dL1_dv8 = v11
dL1_dv6 = v14
dL1_dv4 = v17
------------------------
v18 = v17 + v14
dL1_dL1 = v10
dL1_dv9 = v

In [4]:
# Rules of thumb for Autograd
# 1. Every use of a Variable generates a gradient specific to that use 
# Meaning if we construct a graph where we use a Variable A in two spots ( as in the simple function ) it will ahve a local gradient 
# for both use context - to correctly caclulate its gradient we have to consider ALL uses of the variable

# 2. Inputs become outputs, outputs become inputs, reads become writes, writes become reads
# This addresses a confusion we had earlier - when we record out tape entries, we record them from the inputs from the perspective of the 
# forward pass - so the inputs and outputs of the propogate pass are flipped ( which makes sense -> remember our understanding -> drhs inputs are
# are acually everything that its to the right hand side ( which if we are looking at the dag is the output of the new node r ( or self )
#
# A corrolary here occurs at compute level - because every read of a value in a matrix produces a gradient - it implies in the backward
# pass that we will write a value for every read in the forward - this effectively the intuition we got from our karpathy walkthrough
# that essentially backprop is application of the inverse operation of whatever happened in the forward pass
#
# 3. Every grad call produces grads for a different loss
# grad(l, [a,b]) is computing dl/da and dl/db -> any subsequent call will use a different loss 

In [5]:
# Creating Aggregate Operations
# autograd is great for chaining fundamental operators together, but in some cases you want to create aggregate operators that do not
# perform autograd operations internally. See the following example where we want autograd to compute the entire body (a+b)*b as an aggregate
def simple(a,b):
    t = a + b
    return t* b

def simple_type_error(a,b):
    t = (a.value + b.value)
    r = Variable(t*b.value)
    def propagate(dL_doutputs: List[Variable]) -> List[Variable]:
        # manually apply the chain rule to the compute,
        # in practice a symbolic differentiator might create this code
        dL_dr, = dL_doutputs
        dr_dt = b # partial from: r = t * b
        dr_db = t # partial from: r = t * b
        dL_dt = dL_dr*dr_dt # chain rule
        dt_da = 1.0 # partial from t = a + b
        dt_db = 1.0 # partial from t = a + b
        dL_da = dL_dt * dt_da # chain rule
        #       (contribution from b*t) + (contribution from the b at the leaf node of ( a+b)
        # HOWEVER
        dL_db = dL_dt * dt_db + dL_dr * dr_db # ERROR! dr_db is a Tensor not a Variable
        return [dL_da, dL_db]
    gradient_tape.append(TapeEntry(inputs=[a.name, b.name], outputs=[r.name], propagate=propagate))
    return r

da, db = run_gradients(simple_type_error)
# The reason this doesn't work is because propagate expects to work on variables - which t is not since we've extracted it from autograd


a = tensor([0.2968, 0.7435, 0.0106, 0.0149])
b = tensor([0.7268, 0.8585, 0.7843, 0.6329])
L0 = v0.sum()
dL0 ------------------------
v2 = v1.expand(4)
dL0_dL0 = v1
dL0_dv0 = v2
------------------------
v3 = v2 * b


AttributeError: 'Tensor' object has no attribute 'value'

In [6]:
# To fix this we can potentially recompute t during the call
def simple_recompute(a, b):
    t = (a.value + b.value)
    r = Variable(t * b.value)
    def propagate(dL_doutputs: List[Variable]) -> List[Variable]:
        dL_dr, = dL_doutputs
        dr_dt = b # partial from: r = t * b
        t = a + b # RECOMPUTE!
        dr_db = t # partial from: r = t * b
        dL_dt = dL_dr*dr_dt # chain rule
        dt_da = 1.0 # partial from t = a + b
        dt_db = 1.0 # partial from t = a + b
        dL_da = dL_dt * dt_da # chain rule
        dL_db = dL_dt * dt_db + dL_dr * dr_db # chain rule
        return [dL_da, dL_db]
    gradient_tape.append(TapeEntry(inputs=[a.name, b.name], outputs=[r.name], propagate=propagate))
    return r
da, db = run_gradients(simple_recompute)
print("da", da)
print("db", db)

# which works - however if calculating t was expensive ( such as convolutions or multiplies or what have you ) this would be less than ideal 
# second - we need to save a AND b to get t -> meaning we'd have to waste more memory to compute this as well

a = tensor([0.2968, 0.7435, 0.0106, 0.0149])
b = tensor([0.7268, 0.8585, 0.7843, 0.6329])
L0 = v0.sum()
dL0 ------------------------
v2 = v1.expand(4)
dL0_dL0 = v1
dL0_dv0 = v2
------------------------
v3 = a + b
v4 = v2 * b
v5 = v2 * v3
v6 = v4 + v5
dL0_dL0 = v1
dL0_dv0 = v2
dL0_da = v4
dL0_db = v6
------------------------
v7 = v4 * v4
v8 = v6 * v6
v9 = v7 + v8
L1 = v9.sum()
dL1 ------------------------
v11 = v10.expand(4)
dL1_dL1 = v10
dL1_dv9 = v11
------------------------
dL1_dL1 = v10
dL1_dv9 = v11
dL1_dv7 = v11
dL1_dv8 = v11
------------------------
v12 = v11 * v6
v13 = v11 * v6
v14 = v12 + v13
dL1_dL1 = v10
dL1_dv9 = v11
dL1_dv7 = v11
dL1_dv8 = v11
dL1_dv6 = v14
------------------------
v15 = v11 * v4
v16 = v11 * v4
v17 = v15 + v16
dL1_dL1 = v10
dL1_dv9 = v11
dL1_dv7 = v11
dL1_dv8 = v11
dL1_dv6 = v14
dL1_dv4 = v17
------------------------
v18 = v17 + v14
dL1_dL1 = v10
dL1_dv9 = v11
dL1_dv7 = v11
dL1_dv8 = v11
dL1_dv6 = v14
dL1_dv4 = v18
dL1_dv5 = v14
------------------------
v19

In [7]:
# What if we instead try making t a Variable instead?
def simple_variable_wrong(a, b):
    t = (a.value + b.value)
    t_v = Variable(t, name='t') # named for debugging
    r = Variable(t * b.value)
    def propagate(dL_doutputs: List[Variable]) -> List[Variable]:
        dL_dr, = dL_doutputs
        dr_dt = b # partial from: r = t * b
        dr_db = t_v # partial from: r = t * b
        dL_dt = dL_dr*dr_dt # chain rule
        dt_da = 1.0 # partial from t = a + b
        dt_db = 1.0 # partial from t = a + b
        dL_da = dL_dt * dt_da # chain rule
        dL_db = dL_dt * dt_db + dL_dr * dr_db # chain rule
        return [dL_da, dL_db]
    gradient_tape.append(TapeEntry(inputs=[a.name, b.name], outputs=[r.name], propagate=propagate))
    return r

da, db = run_gradients(simple_variable_wrong)
print("da", da) # ERROR: da is None!!!????
print("db", db)

a = tensor([0.2968, 0.7435, 0.0106, 0.0149])
b = tensor([0.7268, 0.8585, 0.7843, 0.6329])
L0 = v0.sum()
dL0 ------------------------
v2 = v1.expand(4)
dL0_dL0 = v1
dL0_dv0 = v2
------------------------
v3 = v2 * b
v4 = v2 * t
v5 = v3 + v4
dL0_dL0 = v1
dL0_dv0 = v2
dL0_da = v3
dL0_db = v5
------------------------
v6 = v3 * v3
v7 = v5 * v5
v8 = v6 + v7
L1 = v8.sum()
dL1 ------------------------
v10 = v9.expand(4)
dL1_dL1 = v9
dL1_dv8 = v10
------------------------
dL1_dL1 = v9
dL1_dv8 = v10
dL1_dv6 = v10
dL1_dv7 = v10
------------------------
v11 = v10 * v5
v12 = v10 * v5
v13 = v11 + v12
dL1_dL1 = v9
dL1_dv8 = v10
dL1_dv6 = v10
dL1_dv7 = v10
dL1_dv5 = v13
------------------------
v14 = v10 * v3
v15 = v10 * v3
v16 = v14 + v15
dL1_dL1 = v9
dL1_dv8 = v10
dL1_dv6 = v10
dL1_dv7 = v10
dL1_dv5 = v13
dL1_dv3 = v16
------------------------
v17 = v16 + v13
dL1_dL1 = v9
dL1_dv8 = v10
dL1_dv6 = v10
dL1_dv7 = v10
dL1_dv5 = v13
dL1_dv3 = v17
dL1_dv4 = v13
------------------------
v18 = v13 * t
v19 = v

In [8]:
# So something is off - we know da can't be None as it in fact does affect the norm of the gradients
# we aren't properly propagating somewhere ... but where? 
# lets try and troubleshoot by running the first gradient
da, db = run_gradients(simple_variable_wrong, second_loss=False)
da_ref, db_ref = run_gradients(simple, second_loss=False)
print("da", da, da_ref) 
print("db", db, db_ref)

a = tensor([0.2968, 0.7435, 0.0106, 0.0149])
b = tensor([0.7268, 0.8585, 0.7843, 0.6329])
L0 = v0.sum()
dL0 ------------------------
v2 = v1.expand(4)
dL0_dL0 = v1
dL0_dv0 = v2
------------------------
v3 = v2 * b
v4 = v2 * t
v5 = v3 + v4
dL0_dL0 = v1
dL0_dv0 = v2
dL0_da = v3
dL0_db = v5
------------------------
a = tensor([0.2968, 0.7435, 0.0106, 0.0149])
b = tensor([0.7268, 0.8585, 0.7843, 0.6329])
v0 = a + b
v1 = v0 * b
L0 = v1.sum()
dL0 ------------------------
v3 = v2.expand(4)
dL0_dL0 = v2
dL0_dv1 = v3
------------------------
v4 = v3 * b
v5 = v3 * v0
dL0_dL0 = v2
dL0_dv1 = v3
dL0_dv0 = v4
dL0_db = v5
------------------------
v6 = v5 + v4
dL0_dL0 = v2
dL0_dv1 = v3
dL0_dv0 = v4
dL0_db = v6
dL0_da = v4
------------------------
da tensor([0.7268, 0.8585, 0.7843, 0.6329]) tensor([0.7268, 0.8585, 0.7843, 0.6329])
db tensor([1.7504, 2.4605, 1.5791, 1.2808]) tensor([1.7504, 2.4605, 1.5791, 1.2808])


In [ ]:
# which looks correct - lets investigate by looking at the computation trace
# look at the line v4 = v2 * t - it gets used in the first backward - but notice that if another computation uses t - it will have a non
# zero gradient (dL1/dt) for any future loss L1 - that uses the result of that computation. However that future loss calculation
# is not accounted for in our simple_variable_wrong functino -> we don't consider the effects of t outside of the aggregate function
# and thus we lose that portion because t escapes our computations via its usage in the propagate closure -> thus this gradient pathway
# is the only non-zero one as seen from the higher order gradient perspective

In [9]:
# well we can fix that by appending t to the outputs
def simple_variable_almost(a, b):
    t = (a.value + b.value)
    t_v = Variable(t, name='t_v')
    r = Variable(t * b.value)
    def propagate(dL_doutputs: List[Variable]) -> List[Variable]:
        # t is considered an output, so we now get dL_dt0 as an input.
        dL_dr, dL_dt0 = dL_doutputs
               ###### new gradient contribution

        # Handle cases where one incoming gradient is zero (None)
        if dL_dr is None:
          dL_dr = Variable.constant(torch.zeros(()))
        if dL_dt0 is None:
          dL_dt0 = Variable.constant(torch.zeros(()))
               

        dr_dt = b 
        dr_db = t_v 
        # we combine this with the contribution from r to calculate 
        # all gradient paths to dL_dt
        dL_dt = dL_dt0 + dL_dr*dr_dt # chain rule
                ######

        dt_da = 1.0 
        dt_db = 1.0 
        dL_db = dL_dr * dr_db + dL_dt * dt_db 
        dL_da = dL_dt * dt_da
        return [dL_da, dL_db]

    # note: t_v is now considered an output in the tape
    gradient_tape.append(TapeEntry(inputs=[a.name, b.name], outputs=[r.name, t_v.name], propagate=propagate))
                                                                             ######### new output
    return r
da, db = run_gradients(simple_variable_almost)
print("da", da) 
print("db", db)

a = tensor([0.2968, 0.7435, 0.0106, 0.0149])
b = tensor([0.7268, 0.8585, 0.7843, 0.6329])
L0 = v0.sum()
dL0 ------------------------
v2 = v1.expand(4)
dL0_dL0 = v1
dL0_dv0 = v2
------------------------
v3 = 0.0
v4 = v2 * b
v5 = v3 + v4
v6 = v2 * t_v
v7 = v6 + v5
dL0_dL0 = v1
dL0_dv0 = v2
dL0_da = v5
dL0_db = v7
------------------------
v8 = v5 * v5
v9 = v7 * v7
v10 = v8 + v9
L1 = v10.sum()
dL1 ------------------------
v12 = v11.expand(4)
dL1_dL1 = v11
dL1_dv10 = v12
------------------------
dL1_dL1 = v11
dL1_dv10 = v12
dL1_dv8 = v12
dL1_dv9 = v12
------------------------
v13 = v12 * v7
v14 = v12 * v7
v15 = v13 + v14
dL1_dL1 = v11
dL1_dv10 = v12
dL1_dv8 = v12
dL1_dv9 = v12
dL1_dv7 = v15
------------------------
v16 = v12 * v5
v17 = v12 * v5
v18 = v16 + v17
dL1_dL1 = v11
dL1_dv10 = v12
dL1_dv8 = v12
dL1_dv9 = v12
dL1_dv7 = v15
dL1_dv5 = v18
------------------------
v19 = v18 + v15
dL1_dL1 = v11
dL1_dv10 = v12
dL1_dv8 = v12
dL1_dv9 = v12
dL1_dv7 = v15
dL1_dv5 = v19
dL1_dv6 = v15
---------

In [ ]:
# Nice so now the code is correct - however it has some... non optimal behavior:
# we can add a constant - time check for zero in order to elimate a tensor-sized amount of work

In [ ]:
def add_optional(a: Optional['Variable'], b: Optional['Variable']):
    if a is None:
        return b
    if b is None:
        return a
    return a + b

def simple_variable(a, b):
    t = (a.value + b.value)
    t_v = Variable(t, name='t_v')
    r = Variable(t * b.value)
    def propagate(dL_doutputs: List[Variable]) -> List[Variable]:
        dL_dr, dL_dt0 = dL_doutputs
        dr_dt = b # partial from: r = t * b
        dr_db = t_v # partial from: r = t * b
        dL_dt = dL_dt0
        if dL_dr is not None:
            dL_dt = add_optional(dL_dt, dL_dr*dr_dt) # chain rule

        dt_da = 1.0 # partial from t = a + b
        dt_db = 1.0 # partial from t = a + b
        if dL_dr is not None:
            dL_db = dL_dr * dr_db # chain rule
        else:
            dL_db = None

        if dL_dt is not None:
            dL_da = dL_dt * dt_da # chain rule
            dL_db = add_optional(dL_db, dL_dt * dt_db)
        else:
            dL_da = None

        return [dL_da, dL_db]

    gradient_tape.append(TapeEntry(inputs=[a.name, b.name], outputs=[r.name, t_v.name], propagate=propagate))
    return r
da, db = run_gradients(simple_variable)
print("da", da) 
print("db", db)